# Foundation LLM — 50M Parameter GPT

Self-bootstrapping notebook for Google Colab (free T4 GPU).

**Steps:** Runtime → Change runtime type → **GPU** → Run all

Project is created on Google Drive at `MyDrive/foundation-llm/`.

In [ ]:
# Cell 1: GPU check
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime -> Change runtime type -> GPU'
!nvidia-smi
print('GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/foundation-llm')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
%cd {PROJECT_ROOT}
print('Project root:', PROJECT_ROOT)

In [ ]:
# Cell 3: Scaffold project files on Google Drive
import json
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/foundation-llm')
MANIFEST = json.loads('{"config/model_50m_colab.yaml": "# 50M parameter GPT \\u2014 tuned for Google Colab free T4 GPU\\n\\nmodel:\\n  block_size: 1024\\n  vocab_size: 50304\\n  n_layer: 8\\n  n_head: 8\\n  n_embd: 512\\n  dropout: 0.0\\n  bias: false\\n\\ntraining:\\n  batch_size: 8\\n  gradient_accumulation_steps: 16\\n  max_steps: 50000\\n  learning_rate: 3.0e-4\\n  weight_decay: 0.1\\n  beta1: 0.9\\n  beta2: 0.95\\n  grad_clip: 1.0\\n  warmup_steps: 1000\\n  lr_decay_steps: 50000\\n  min_lr: 3.0e-5\\n  eval_interval: 500\\n  log_interval: 10\\n  checkpoint_interval: 1000\\n  dtype: float16\\n\\ndata:\\n  train_bin: data/processed/train.bin\\n  val_bin: data/processed/val.bin\\n  meta_pkl: data/processed/meta.pkl\\n\\npaths:\\n  checkpoint_dir: checkpoints\\n  cache_dir: .cache\\n", "requirements.txt": "torch>=2.1.0\\ntiktoken>=0.5.0\\ndatasets>=2.14.0\\npyyaml>=6.0\\ntqdm>=4.66.0\\nnumpy>=1.24.0\\n", "data/sample_corpus.txt": "The history of artificial intelligence begins with the dream of creating machines that can think. Early pioneers imagined mechanical brains that could solve puzzles, play games, and understand language. Over decades, researchers built systems that could recognize patterns, prove theorems, and eventually learn from data.\\n\\nMachine learning changed the direction of the field. Instead of hand-coding every rule, engineers trained models on examples. Neural networks grew deeper and wider, absorbing vast collections of text, images, and speech. Each generation of models learned richer representations of the world.\\n\\nLanguage models became a central tool for AI research. By predicting the next word in a sentence, a model learns grammar, facts, and reasoning patterns. Small models can capture local structure, while larger models generalize across many topics. Training requires careful tuning of learning rates, batch sizes, and data quality.\\n\\nFoundation models are pretrained on broad data and later adapted to specific tasks. They serve as a base layer of knowledge that can be fine-tuned for chat, coding, search, and scientific discovery. Building such a model from scratch teaches the full stack: tokenization, architecture design, optimization, and evaluation.\\n\\nNatural language processing relies on tokenizers to split text into subword units. Byte-pair encoding balances vocabulary size with coverage of rare words. Once text is tokenized, models consume sequences of integers and learn embeddings that map tokens into continuous vectors.\\n\\nTransformers replaced older recurrent architectures for many language tasks. Self-attention lets each token attend to previous tokens in a sequence, capturing long-range dependencies efficiently. Stacked transformer blocks with feed-forward layers form the backbone of modern decoder-only language models.\\n\\nTraining a language model means minimizing cross-entropy loss over millions or billions of tokens. Gradients flow backward through the network, updating weights to make better predictions. Mixed precision training reduces memory usage and speeds up computation on modern GPUs.\\n\\nEvaluation uses held-out validation data to estimate perplexity, which measures how surprised the model is by unseen text. Lower perplexity generally indicates better fit, though human judgment remains important for assessing fluency and usefulness.\\n\\nGeneration samples text autoregressively, one token at a time. Temperature and top-k sampling control randomness, trading repetition for creativity. Even small models can produce readable paragraphs after sufficient training on diverse corpora.\\n\\nOpen datasets such as web text collections provide the raw material for pretraining. Filtering, deduplication, and quality heuristics improve the signal-to-noise ratio. Responsible dataset curation reduces harmful content and respects licensing constraints.\\n\\nCheckpoints save model weights during training so experiments can resume after interruptions. This is essential in cloud notebooks where sessions expire. Saving optimizer state allows seamless continuation without losing momentum estimates.\\n\\nThe path from random initialization to coherent language is long but instructive. Each training step nudges parameters toward patterns present in the data. With patience, careful engineering, and enough tokens, a compact foundation model can learn to write, summarize, and answer simple prompts.\\n\\nScience fiction once portrayed intelligent machines as distant fantasies. Today, language models assist writers, programmers, and students every day. Understanding how they are built demystifies their strengths and limitations.\\n\\nEducation benefits when learners can experiment with small models locally or in the cloud. A fifty-million-parameter model fits on a single GPU and trains in hours or days rather than months. That scale is ideal for teaching the fundamentals without datacenter budgets.\\n\\nFuture work may add instruction tuning, reinforcement learning from human feedback, or multimodal inputs. The foundation remains the same: predict tokens, learn representations, evaluate honestly, and iterate. Every large model begins as a small experiment that someone chose to run carefully.\\n\\nArtificial intelligence continues to evolve rapidly. New architectures, datasets, and training tricks appear each year. Yet the core loop of data, model, loss, and optimization endures. Mastering that loop is the first step toward building systems that genuinely help people.\\n", "data/prepare.py": "\\"\\"\\"Prepare tokenized training data from text corpora.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport os\\nimport pickle\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport tiktoken\\nfrom tqdm import tqdm\\n\\n\\ndef get_project_root() -> Path:\\n    return Path(__file__).resolve().parent.parent\\n\\n\\ndef setup_cache_dirs(project_root: Path) -> None:\\n    cache_path = project_root / \\".cache\\"\\n    cache_path.mkdir(parents=True, exist_ok=True)\\n    os.environ[\\"HF_HOME\\"] = str(cache_path / \\"huggingface\\")\\n    os.environ[\\"TRANSFORMERS_CACHE\\"] = str(cache_path / \\"huggingface\\")\\n\\n\\ndef tokenize_texts(texts: list[str], enc: tiktoken.Encoding) -> np.ndarray:\\n    ids: list[int] = []\\n    for text in tqdm(texts, desc=\\"Tokenizing\\"):\\n        ids.extend(enc.encode_ordinary(text))\\n        ids.append(enc.eot_token)\\n    return np.array(ids, dtype=np.uint16)\\n\\n\\ndef write_bin(path: Path, tokens: np.ndarray) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    tokens.tofile(path)\\n    print(f\\"Wrote {path} ({len(tokens):,} tokens, {path.stat().st_size / 1e6:.1f} MB)\\")\\n\\n\\ndef prepare_from_sample(project_root: Path, val_ratio: float = 0.1) -> None:\\n    sample_path = project_root / \\"data\\" / \\"sample_corpus.txt\\"\\n    if not sample_path.exists():\\n        raise FileNotFoundError(f\\"Sample corpus not found: {sample_path}\\")\\n\\n    enc = tiktoken.get_encoding(\\"gpt2\\")\\n    text = sample_path.read_text(encoding=\\"utf-8\\")\\n    paragraphs = [p.strip() for p in text.split(\\"\\\\n\\\\n\\") if p.strip()]\\n    split_idx = max(1, int(len(paragraphs) * (1 - val_ratio)))\\n    train_texts = paragraphs[:split_idx]\\n    val_texts = paragraphs[split_idx:] or paragraphs[-1:]\\n\\n    train_ids = tokenize_texts(train_texts, enc)\\n    val_ids = tokenize_texts(val_texts, enc)\\n\\n    out_dir = project_root / \\"data\\" / \\"processed\\"\\n    write_bin(out_dir / \\"train.bin\\", train_ids)\\n    write_bin(out_dir / \\"val.bin\\", val_ids)\\n\\n    meta = {\\n        \\"vocab_size\\": enc.n_vocab,\\n        \\"encoding_name\\": \\"gpt2\\",\\n        \\"source\\": \\"sample_corpus\\",\\n    }\\n    with (out_dir / \\"meta.pkl\\").open(\\"wb\\") as f:\\n        pickle.dump(meta, f)\\n    print(\\"Sample data preparation complete.\\")\\n\\n\\ndef prepare_openwebtext(project_root: Path, max_docs: int, val_ratio: float = 0.1) -> None:\\n    from datasets import load_dataset\\n\\n    setup_cache_dirs(project_root)\\n    enc = tiktoken.get_encoding(\\"gpt2\\")\\n\\n    print(f\\"Streaming OpenWebText subset (max_docs={max_docs:,})...\\")\\n    dataset = load_dataset(\\"Skylion007/openwebtext\\", split=\\"train\\", streaming=True)\\n\\n    texts: list[str] = []\\n    for i, row in enumerate(tqdm(dataset, total=max_docs, desc=\\"Loading docs\\")):\\n        if i >= max_docs:\\n            break\\n        text = row.get(\\"text\\", \\"\\").strip()\\n        if len(text) > 100:\\n            texts.append(text)\\n\\n    if len(texts) < 10:\\n        raise RuntimeError(\\"Not enough documents loaded from OpenWebText\\")\\n\\n    split_idx = max(1, int(len(texts) * (1 - val_ratio)))\\n    train_texts = texts[:split_idx]\\n    val_texts = texts[split_idx:] or texts[-max(1, len(texts) // 10) :]\\n\\n    train_ids = tokenize_texts(train_texts, enc)\\n    val_ids = tokenize_texts(val_texts, enc)\\n\\n    out_dir = project_root / \\"data\\" / \\"processed\\"\\n    write_bin(out_dir / \\"train.bin\\", train_ids)\\n    write_bin(out_dir / \\"val.bin\\", val_ids)\\n\\n    meta = {\\n        \\"vocab_size\\": enc.n_vocab,\\n        \\"encoding_name\\": \\"gpt2\\",\\n        \\"source\\": \\"openwebtext\\",\\n        \\"max_docs\\": max_docs,\\n    }\\n    with (out_dir / \\"meta.pkl\\").open(\\"wb\\") as f:\\n        pickle.dump(meta, f)\\n    print(\\"OpenWebText data preparation complete.\\")\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=\\"Prepare tokenized training data\\")\\n    parser.add_argument(\\"--sample\\", action=\\"store_true\\", help=\\"Use bundled sample corpus\\")\\n    parser.add_argument(\\"--subset\\", action=\\"store_true\\", help=\\"Use OpenWebText subset\\")\\n    parser.add_argument(\\"--max_docs\\", type=int, default=200000, help=\\"Max OpenWebText documents\\")\\n    args = parser.parse_args()\\n\\n    project_root = get_project_root()\\n    if args.sample:\\n        prepare_from_sample(project_root)\\n    elif args.subset:\\n        prepare_openwebtext(project_root, max_docs=args.max_docs)\\n    else:\\n        prepare_openwebtext(project_root, max_docs=args.max_docs)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "src/__init__.py": "", "src/model.py": "\\"\\"\\"GPT-style decoder-only transformer (~50M parameters).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nfrom dataclasses import dataclass\\n\\nimport torch\\nimport torch.nn as nn\\nfrom torch.nn import functional as F\\n\\n\\n@dataclass\\nclass GPTConfig:\\n    block_size: int = 1024\\n    vocab_size: int = 50304\\n    n_layer: int = 8\\n    n_head: int = 8\\n    n_embd: int = 512\\n    dropout: float = 0.0\\n    bias: bool = False\\n\\n\\nclass CausalSelfAttention(nn.Module):\\n    def __init__(self, config: GPTConfig) -> None:\\n        super().__init__()\\n        assert config.n_embd % config.n_head == 0\\n        self.n_head = config.n_head\\n        self.n_embd = config.n_embd\\n        self.dropout = config.dropout\\n        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)\\n        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)\\n        self.attn_dropout = nn.Dropout(config.dropout)\\n        self.resid_dropout = nn.Dropout(config.dropout)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        batch, seq_len, channels = x.size()\\n        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)\\n        head_size = channels // self.n_head\\n        q = q.view(batch, seq_len, self.n_head, head_size).transpose(1, 2)\\n        k = k.view(batch, seq_len, self.n_head, head_size).transpose(1, 2)\\n        v = v.view(batch, seq_len, self.n_head, head_size).transpose(1, 2)\\n\\n        if hasattr(F, \\"scaled_dot_product_attention\\"):\\n            y = F.scaled_dot_product_attention(\\n                q,\\n                k,\\n                v,\\n                attn_mask=None,\\n                dropout_p=self.dropout if self.training else 0.0,\\n                is_causal=True,\\n            )\\n        else:\\n            att = (q @ k.transpose(-2, -1)) / math.sqrt(head_size)\\n            mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool))\\n            att = att.masked_fill(~mask, float(\\"-inf\\"))\\n            att = F.softmax(att, dim=-1)\\n            att = self.attn_dropout(att)\\n            y = att @ v\\n\\n        y = y.transpose(1, 2).contiguous().view(batch, seq_len, channels)\\n        y = self.resid_dropout(self.c_proj(y))\\n        return y\\n\\n\\nclass MLP(nn.Module):\\n    def __init__(self, config: GPTConfig) -> None:\\n        super().__init__()\\n        hidden = 4 * config.n_embd\\n        self.c_fc = nn.Linear(config.n_embd, hidden, bias=config.bias)\\n        self.gelu = nn.GELU()\\n        self.c_proj = nn.Linear(hidden, config.n_embd, bias=config.bias)\\n        self.dropout = nn.Dropout(config.dropout)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        x = self.c_fc(x)\\n        x = self.gelu(x)\\n        x = self.c_proj(x)\\n        return self.dropout(x)\\n\\n\\nclass Block(nn.Module):\\n    def __init__(self, config: GPTConfig) -> None:\\n        super().__init__()\\n        self.ln_1 = nn.LayerNorm(config.n_embd)\\n        self.attn = CausalSelfAttention(config)\\n        self.ln_2 = nn.LayerNorm(config.n_embd)\\n        self.mlp = MLP(config)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        x = x + self.attn(self.ln_1(x))\\n        x = x + self.mlp(self.ln_2(x))\\n        return x\\n\\n\\nclass GPT(nn.Module):\\n    def __init__(self, config: GPTConfig) -> None:\\n        super().__init__()\\n        self.config = config\\n        self.transformer = nn.ModuleDict(\\n            {\\n                \\"wte\\": nn.Embedding(config.vocab_size, config.n_embd),\\n                \\"wpe\\": nn.Embedding(config.block_size, config.n_embd),\\n                \\"drop\\": nn.Dropout(config.dropout),\\n                \\"h\\": nn.ModuleList([Block(config) for _ in range(config.n_layer)]),\\n                \\"ln_f\\": nn.LayerNorm(config.n_embd),\\n            }\\n        )\\n        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)\\n        self.transformer.wte.weight = self.lm_head.weight\\n        self.apply(self._init_weights)\\n\\n    def _init_weights(self, module: nn.Module) -> None:\\n        if isinstance(module, nn.Linear):\\n            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)\\n            if module.bias is not None:\\n                torch.nn.init.zeros_(module.bias)\\n        elif isinstance(module, nn.Embedding):\\n            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)\\n\\n    def forward(\\n        self,\\n        idx: torch.Tensor,\\n        targets: torch.Tensor | None = None,\\n    ) -> tuple[torch.Tensor, torch.Tensor | None]:\\n        _, seq_len = idx.size()\\n        if seq_len > self.config.block_size:\\n            raise ValueError(f\\"Sequence length {seq_len} exceeds block_size {self.config.block_size}\\")\\n\\n        pos = torch.arange(0, seq_len, dtype=torch.long, device=idx.device)\\n        x = self.transformer.wte(idx) + self.transformer.wpe(pos)\\n        x = self.transformer.drop(x)\\n        for block in self.transformer.h:\\n            x = block(x)\\n        x = self.transformer.ln_f(x)\\n        logits = self.lm_head(x)\\n\\n        loss = None\\n        if targets is not None:\\n            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))\\n        return logits, loss\\n\\n    @torch.no_grad()\\n    def generate(\\n        self,\\n        idx: torch.Tensor,\\n        max_new_tokens: int,\\n        temperature: float = 1.0,\\n        top_k: int | None = None,\\n    ) -> torch.Tensor:\\n        for _ in range(max_new_tokens):\\n            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size :]\\n            logits, _ = self(idx_cond)\\n            logits = logits[:, -1, :] / max(temperature, 1e-8)\\n            if top_k is not None:\\n                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))\\n                logits[logits < values[:, [-1]]] = float(\\"-inf\\")\\n            probs = F.softmax(logits, dim=-1)\\n            next_token = torch.multinomial(probs, num_samples=1)\\n            idx = torch.cat((idx, next_token), dim=1)\\n        return idx\\n\\n    def count_parameters(self) -> int:\\n        return sum(p.numel() for p in self.parameters())\\n\\n\\ndef build_model(config: GPTConfig) -> GPT:\\n    model = GPT(config)\\n    params = model.count_parameters()\\n    target = 50_000_000\\n    if not (0.90 * target <= params <= 1.10 * target):\\n        print(f\\"Warning: parameter count {params:,} is outside 45M-55M target range\\")\\n    else:\\n        print(f\\"Model parameters: {params:,} (~50M)\\")\\n    return model\\n", "src/dataset.py": "\\"\\"\\"Memory-mapped token dataset for language model training.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom torch.utils.data import Dataset\\n\\n\\nclass TokenBinDataset(Dataset):\\n    def __init__(self, bin_path: str | Path, block_size: int) -> None:\\n        self.block_size = block_size\\n        self.data = np.memmap(Path(bin_path), dtype=np.uint16, mode=\\"r\\")\\n        if len(self.data) <= block_size + 1:\\n            raise ValueError(\\n                f\\"Dataset at {bin_path} is too small ({len(self.data)} tokens) \\"\\n                f\\"for block_size={block_size}\\"\\n            )\\n\\n    def __len__(self) -> int:\\n        return len(self.data) - self.block_size\\n\\n    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:\\n        chunk = self.data[idx : idx + self.block_size + 1].astype(np.int64)\\n        x = torch.from_numpy(chunk[:-1].copy())\\n        y = torch.from_numpy(chunk[1:].copy())\\n        return x, y\\n", "src/train.py": "\\"\\"\\"Training loop for the 50M GPT foundation model.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport math\\nimport os\\nimport pickle\\nimport time\\nfrom pathlib import Path\\n\\nimport torch\\nimport yaml\\nfrom torch.utils.data import DataLoader\\n\\nfrom dataset import TokenBinDataset\\nfrom model import GPT, GPTConfig, build_model\\n\\n\\ndef get_project_root() -> Path:\\n    return Path(__file__).resolve().parent.parent\\n\\n\\ndef setup_cache_dirs(project_root: Path, cache_dir: str) -> None:\\n    cache_path = project_root / cache_dir\\n    cache_path.mkdir(parents=True, exist_ok=True)\\n    os.environ[\\"HF_HOME\\"] = str(cache_path / \\"huggingface\\")\\n    os.environ[\\"TRANSFORMERS_CACHE\\"] = str(cache_path / \\"huggingface\\")\\n    os.environ[\\"TORCH_HOME\\"] = str(cache_path / \\"torch\\")\\n\\n\\ndef load_config(config_path: Path) -> dict:\\n    with config_path.open(\\"r\\", encoding=\\"utf-8\\") as f:\\n        return yaml.safe_load(f)\\n\\n\\ndef get_lr(step: int, cfg: dict) -> float:\\n    base_lr = cfg[\\"learning_rate\\"]\\n    warmup = cfg[\\"warmup_steps\\"]\\n    decay_steps = cfg[\\"lr_decay_steps\\"]\\n    min_lr = cfg[\\"min_lr\\"]\\n\\n    if step < warmup:\\n        return base_lr * step / max(warmup, 1)\\n    if step >= decay_steps:\\n        return min_lr\\n    decay_ratio = (step - warmup) / max(decay_steps - warmup, 1)\\n    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))\\n    return min_lr + coeff * (base_lr - min_lr)\\n\\n\\ndef find_latest_checkpoint(checkpoint_dir: Path) -> Path | None:\\n    if not checkpoint_dir.exists():\\n        return None\\n    checkpoints = sorted(checkpoint_dir.glob(\\"step_*.pt\\"))\\n    if checkpoints:\\n        return checkpoints[-1]\\n    latest = checkpoint_dir / \\"latest.pt\\"\\n    return latest if latest.exists() else None\\n\\n\\ndef save_checkpoint(\\n    checkpoint_dir: Path,\\n    step: int,\\n    model: GPT,\\n    optimizer: torch.optim.Optimizer,\\n    scaler: torch.cuda.amp.GradScaler | None,\\n    best_val_loss: float,\\n) -> None:\\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\\n    payload = {\\n        \\"step\\": step,\\n        \\"model\\": model.state_dict(),\\n        \\"optimizer\\": optimizer.state_dict(),\\n        \\"best_val_loss\\": best_val_loss,\\n        \\"scaler\\": scaler.state_dict() if scaler is not None else None,\\n    }\\n    step_path = checkpoint_dir / f\\"step_{step:07d}.pt\\"\\n    latest_path = checkpoint_dir / \\"latest.pt\\"\\n    torch.save(payload, step_path)\\n    torch.save(payload, latest_path)\\n    print(f\\"Saved checkpoint: {step_path}\\")\\n\\n\\ndef evaluate(model: GPT, loader: DataLoader, device: torch.device, dtype: torch.dtype) -> float:\\n    model.eval()\\n    total_loss = 0.0\\n    total_tokens = 0\\n    with torch.no_grad():\\n        for x, y in loader:\\n            x = x.to(device, non_blocking=True)\\n            y = y.to(device, non_blocking=True)\\n            with torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == \\"cuda\\"):\\n                _, loss = model(x, y)\\n            tokens = y.numel()\\n            total_loss += loss.item() * tokens\\n            total_tokens += tokens\\n    model.train()\\n    return total_loss / max(total_tokens, 1)\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=\\"Train 50M GPT foundation model\\")\\n    parser.add_argument(\\"--config\\", type=str, default=\\"config/model_50m_colab.yaml\\")\\n    parser.add_argument(\\"--max_steps\\", type=int, default=None)\\n    parser.add_argument(\\"--resume\\", type=str, default=None, help=\\"Path to checkpoint or \'auto\'\\")\\n    args = parser.parse_args()\\n\\n    project_root = get_project_root()\\n    config = load_config(project_root / args.config)\\n    setup_cache_dirs(project_root, config[\\"paths\\"][\\"cache_dir\\"])\\n\\n    train_cfg = config[\\"training\\"]\\n    if args.max_steps is not None:\\n        train_cfg[\\"max_steps\\"] = args.max_steps\\n\\n    device = torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n    dtype_name = train_cfg.get(\\"dtype\\", \\"float16\\")\\n    dtype = torch.float16 if dtype_name == \\"float16\\" else torch.bfloat16\\n    print(f\\"Using device: {device}\\")\\n\\n    model_cfg = GPTConfig(**config[\\"model\\"])\\n    model = build_model(model_cfg).to(device)\\n    if device.type == \\"cuda\\":\\n        torch.backends.cuda.matmul.allow_tf32 = True\\n\\n    train_ds = TokenBinDataset(project_root / config[\\"data\\"][\\"train_bin\\"], model_cfg.block_size)\\n    val_ds = TokenBinDataset(project_root / config[\\"data\\"][\\"val_bin\\"], model_cfg.block_size)\\n    train_loader = DataLoader(\\n        train_ds,\\n        batch_size=train_cfg[\\"batch_size\\"],\\n        shuffle=True,\\n        pin_memory=device.type == \\"cuda\\",\\n        drop_last=True,\\n    )\\n    val_loader = DataLoader(\\n        val_ds,\\n        batch_size=train_cfg[\\"batch_size\\"],\\n        shuffle=False,\\n        pin_memory=device.type == \\"cuda\\",\\n        drop_last=False,\\n    )\\n\\n    optimizer = torch.optim.AdamW(\\n        model.parameters(),\\n        lr=train_cfg[\\"learning_rate\\"],\\n        betas=(train_cfg[\\"beta1\\"], train_cfg[\\"beta2\\"]),\\n        weight_decay=train_cfg[\\"weight_decay\\"],\\n    )\\n    scaler = torch.cuda.amp.GradScaler(enabled=device.type == \\"cuda\\" and dtype == torch.float16)\\n\\n    checkpoint_dir = project_root / config[\\"paths\\"][\\"checkpoint_dir\\"]\\n    start_step = 0\\n    best_val_loss = float(\\"inf\\")\\n\\n    resume_path = None\\n    if args.resume == \\"auto\\":\\n        resume_path = find_latest_checkpoint(checkpoint_dir)\\n    elif args.resume:\\n        resume_path = Path(args.resume)\\n        if not resume_path.is_absolute():\\n            resume_path = project_root / resume_path\\n\\n    if resume_path and resume_path.exists():\\n        print(f\\"Resuming from {resume_path}\\")\\n        ckpt = torch.load(resume_path, map_location=device)\\n        model.load_state_dict(ckpt[\\"model\\"])\\n        optimizer.load_state_dict(ckpt[\\"optimizer\\"])\\n        start_step = ckpt.get(\\"step\\", 0)\\n        best_val_loss = ckpt.get(\\"best_val_loss\\", float(\\"inf\\"))\\n        if scaler is not None and ckpt.get(\\"scaler\\"):\\n            scaler.load_state_dict(ckpt[\\"scaler\\"])\\n\\n    model.train()\\n    train_iter = iter(train_loader)\\n    tokens_per_step = (\\n        train_cfg[\\"batch_size\\"] * model_cfg.block_size * train_cfg[\\"gradient_accumulation_steps\\"]\\n    )\\n    print(f\\"Tokens per optimizer step: {tokens_per_step:,}\\")\\n\\n    for step in range(start_step, train_cfg[\\"max_steps\\"]):\\n        lr = get_lr(step, train_cfg)\\n        for param_group in optimizer.param_groups:\\n            param_group[\\"lr\\"] = lr\\n\\n        optimizer.zero_grad(set_to_none=True)\\n        loss_accum = 0.0\\n        for _ in range(train_cfg[\\"gradient_accumulation_steps\\"]):\\n            try:\\n                x, y = next(train_iter)\\n            except StopIteration:\\n                train_iter = iter(train_loader)\\n                x, y = next(train_iter)\\n\\n            x = x.to(device, non_blocking=True)\\n            y = y.to(device, non_blocking=True)\\n            with torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == \\"cuda\\"):\\n                _, loss = model(x, y)\\n                loss = loss / train_cfg[\\"gradient_accumulation_steps\\"]\\n            if scaler is not None:\\n                scaler.scale(loss).backward()\\n            else:\\n                loss.backward()\\n            loss_accum += loss.item()\\n\\n        if scaler is not None:\\n            scaler.unscale_(optimizer)\\n        torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg[\\"grad_clip\\"])\\n        if scaler is not None:\\n            scaler.step(optimizer)\\n            scaler.update()\\n        else:\\n            optimizer.step()\\n\\n        if step % train_cfg[\\"log_interval\\"] == 0:\\n            print(f\\"step {step:6d} | loss {loss_accum:.4f} | lr {lr:.2e}\\")\\n\\n        if step > 0 and step % train_cfg[\\"eval_interval\\"] == 0:\\n            val_loss = evaluate(model, val_loader, device, dtype)\\n            val_ppl = math.exp(val_loss)\\n            print(f\\"step {step:6d} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}\\")\\n            if val_loss < best_val_loss:\\n                best_val_loss = val_loss\\n\\n        if step > 0 and step % train_cfg[\\"checkpoint_interval\\"] == 0:\\n            save_checkpoint(checkpoint_dir, step, model, optimizer, scaler, best_val_loss)\\n\\n    save_checkpoint(checkpoint_dir, train_cfg[\\"max_steps\\"] - 1, model, optimizer, scaler, best_val_loss)\\n    print(\\"Training complete.\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "src/evaluate.py": "\\"\\"\\"Validation perplexity evaluation.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport math\\nimport pickle\\nfrom pathlib import Path\\n\\nimport torch\\nimport yaml\\nfrom torch.utils.data import DataLoader\\n\\nfrom dataset import TokenBinDataset\\nfrom model import GPT, GPTConfig\\n\\n\\ndef load_config(config_path: Path) -> dict:\\n    with config_path.open(\\"r\\", encoding=\\"utf-8\\") as f:\\n        return yaml.safe_load(f)\\n\\n\\ndef evaluate(\\n    model: GPT,\\n    loader: DataLoader,\\n    device: torch.device,\\n    dtype: torch.dtype,\\n) -> float:\\n    model.eval()\\n    total_loss = 0.0\\n    total_tokens = 0\\n    with torch.no_grad():\\n        for x, y in loader:\\n            x = x.to(device)\\n            y = y.to(device)\\n            with torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == \\"cuda\\"):\\n                _, loss = model(x, y)\\n            tokens = y.numel()\\n            total_loss += loss.item() * tokens\\n            total_tokens += tokens\\n    return math.exp(total_loss / max(total_tokens, 1))\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=\\"Evaluate validation perplexity\\")\\n    parser.add_argument(\\"--config\\", type=str, default=\\"config/model_50m_colab.yaml\\")\\n    parser.add_argument(\\"--checkpoint\\", type=str, required=True)\\n    args = parser.parse_args()\\n\\n    project_root = Path(__file__).resolve().parent.parent\\n    config = load_config(project_root / args.config)\\n    meta_path = project_root / config[\\"data\\"][\\"meta_pkl\\"]\\n    with meta_path.open(\\"rb\\") as f:\\n        meta = pickle.load(f)\\n\\n    device = torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n    dtype_name = config[\\"training\\"].get(\\"dtype\\", \\"float16\\")\\n    dtype = torch.float16 if dtype_name == \\"float16\\" else torch.bfloat16\\n\\n    model_cfg = GPTConfig(**config[\\"model\\"])\\n    model = GPT(model_cfg).to(device)\\n    checkpoint = torch.load(project_root / args.checkpoint, map_location=device)\\n    model.load_state_dict(checkpoint[\\"model\\"])\\n    model.eval()\\n\\n    val_ds = TokenBinDataset(project_root / config[\\"data\\"][\\"val_bin\\"], model_cfg.block_size)\\n    val_loader = DataLoader(val_ds, batch_size=config[\\"training\\"][\\"batch_size\\"], shuffle=False)\\n\\n    ppl = evaluate(model, val_loader, device, dtype)\\n    print(f\\"Validation perplexity: {ppl:.2f}\\")\\n    print(f\\"Vocab size: {meta[\'vocab_size\']:,}\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "src/generate.py": "\\"\\"\\"Text generation from a trained checkpoint.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport pickle\\nfrom pathlib import Path\\n\\nimport torch\\nimport yaml\\n\\nfrom model import GPT, GPTConfig\\n\\n\\ndef load_config(config_path: Path) -> dict:\\n    with config_path.open(\\"r\\", encoding=\\"utf-8\\") as f:\\n        return yaml.safe_load(f)\\n\\n\\ndef encode_text(text: str, meta: dict) -> list[int]:\\n    if meta.get(\\"encoding_name\\"):\\n        import tiktoken\\n\\n        enc = tiktoken.get_encoding(meta[\\"encoding_name\\"])\\n        return enc.encode(text)\\n    raise ValueError(\\"Tokenizer metadata missing encoding_name\\")\\n\\n\\ndef decode_tokens(tokens: list[int], meta: dict) -> str:\\n    import tiktoken\\n\\n    enc = tiktoken.get_encoding(meta[\\"encoding_name\\"])\\n    return enc.decode(tokens)\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=\\"Generate text from a checkpoint\\")\\n    parser.add_argument(\\"--config\\", type=str, default=\\"config/model_50m_colab.yaml\\")\\n    parser.add_argument(\\"--checkpoint\\", type=str, default=\\"checkpoints/latest.pt\\")\\n    parser.add_argument(\\"--prompt\\", type=str, default=\\"Once upon a time\\")\\n    parser.add_argument(\\"--max_tokens\\", type=int, default=200)\\n    parser.add_argument(\\"--temperature\\", type=float, default=0.8)\\n    parser.add_argument(\\"--top_k\\", type=int, default=200)\\n    args = parser.parse_args()\\n\\n    project_root = Path(__file__).resolve().parent.parent\\n    config = load_config(project_root / args.config)\\n    meta_path = project_root / config[\\"data\\"][\\"meta_pkl\\"]\\n    with meta_path.open(\\"rb\\") as f:\\n        meta = pickle.load(f)\\n\\n    device = torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n    model_cfg = GPTConfig(**config[\\"model\\"])\\n    model = GPT(model_cfg).to(device)\\n    checkpoint = torch.load(project_root / args.checkpoint, map_location=device)\\n    model.load_state_dict(checkpoint[\\"model\\"])\\n    model.eval()\\n\\n    start_ids = encode_text(args.prompt, meta)\\n    idx = torch.tensor([start_ids], dtype=torch.long, device=device)\\n    output = model.generate(idx, max_new_tokens=args.max_tokens, temperature=args.temperature, top_k=args.top_k)\\n    text = decode_tokens(output[0].tolist(), meta)\\n    print(\\"\\\\n--- Generated Text ---\\\\n\\")\\n    print(text)\\n    print(\\"\\\\n----------------------\\\\n\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')

written = 0
for rel_path, content in MANIFEST.items():
    out = PROJECT_ROOT / rel_path
    out.parent.mkdir(parents=True, exist_ok=True)
    if not out.exists() or out.read_text(encoding='utf-8') != content:
        out.write_text(content, encoding='utf-8')
        written += 1

for sub in ['checkpoints', 'data/processed', '.cache']:
    (PROJECT_ROOT / sub).mkdir(parents=True, exist_ok=True)

print(f'Scaffold complete. Updated {written} files at {PROJECT_ROOT}')

In [ ]:
# Cell 4: Install dependencies
!pip install -q -r requirements.txt
import torch, tiktoken, yaml
print('Dependencies installed. PyTorch:', torch.__version__)

In [ ]:
# Cell 5: Smoke test (sample data + 10 training steps)
!python data/prepare.py --sample

import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/foundation-llm')
result = subprocess.run(
    [sys.executable, 'src/train.py', '--config', 'config/model_50m_colab.yaml', '--max_steps', '10'],
    cwd=PROJECT_ROOT,
    check=False,
)
if result.returncode != 0:
    raise RuntimeError('Smoke test training failed')

result = subprocess.run(
    [sys.executable, 'src/generate.py', '--checkpoint', 'checkpoints/latest.pt', '--prompt', 'The history of AI', '--max_tokens', '50'],
    cwd=PROJECT_ROOT,
    check=False,
)
print('\n=== SMOKE TEST PASSED ===' if result.returncode == 0 else 'Generation failed')

In [ ]:
# Cell 6: Prepare OpenWebText subset (first full run only; skip if train.bin exists)
from pathlib import Path
train_bin = Path('data/processed/train.bin')
meta = Path('data/processed/meta.pkl')
if train_bin.exists() and meta.exists():
    print('Processed data already exists. Skipping download.')
    print(f'train.bin size: {train_bin.stat().st_size / 1e9:.2f} GB')
else:
    !python data/prepare.py --subset --max_docs 200000

In [ ]:
# Cell 7: Train 50M model (auto-resumes from Drive checkpoints)
!python src/train.py --config config/model_50m_colab.yaml --resume auto

In [ ]:
# Cell 8: Generate sample text
!python src/generate.py --checkpoint checkpoints/latest.pt --prompt "Once upon a time" --max_tokens 200 --temperature 0.8